# Notebook 05: Visualizations

End-to-end visualization for one pitcher, one game, one pitch type.

Uses `build_game_report()` as the single service call — returns a `GameReport` with
all data pre-computed and ready for plotting.

Charts produced:
1. Full multi-panel dashboard
2. Rolling horizontal + vertical spread
3. Rolling miss distance from location hub
4. Location hub cluster map
5. Miss vectors (direction arrows)
6. Cluster drift over time
7. Cluster usage by pitch

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt

from src.services.pitcher_service import build_game_report, get_available_games, get_pitcher_dataset
from src.visualization.dashboards import plot_game_dashboard
from src.visualization.rolling_plots import (
    prep_rolling_spreads, plot_rolling_spreads,
    prep_rolling_miss, plot_rolling_miss,
)
from src.visualization.zone_plots import (
    prep_cluster_map, plot_cluster_map,
    prep_miss_vectors, plot_miss_vectors,
    prep_cluster_drift, plot_cluster_drift,
    prep_cluster_usage, plot_cluster_usage,
)
from src import config

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

In [ ]:
# ---- Parameters ----
PITCHER    = "Gerrit Cole"
START_DATE = "2024-04-01"
END_DATE   = "2024-09-30"
PITCH_TYPE = "FF"
WINDOW     = 7
K          = 4
GAME_PK    = None   # Set to an integer, or None to use first available
# --------------------

# Peek at available games to pick one
if GAME_PK is None:
    df_tmp = get_pitcher_dataset(PITCHER, START_DATE, END_DATE)
    games = get_available_games(df_tmp)
    GAME_PK = games[0].game_pk
    print(f"Selected game: {games[0]}")

## Build full game report (one call)

In [ ]:
report = build_game_report(
    pitcher_id=PITCHER,
    game_pk=GAME_PK,
    pitch_type=PITCH_TYPE,
    start_date=START_DATE,
    end_date=END_DATE,
    window=WINDOW,
    k=K,
)

if report.warnings:
    for w in report.warnings:
        print(f"[WARNING] {w}")

print(f"Report ready: pitcher={report.pitcher_id}, game={report.game_pk}, "
      f"pitch_type={report.pitch_type}, date={report.game_date}")
print(f"Game pitches: {len(report.game_with_clusters)}, "
      f"hubs: {report.cluster_model.k}, "
      f"overall drift: {report.drift_report.overall_drift_magnitude:.3f} ft")

## Chart 1: Multi-panel dashboard

In [ ]:
fig = plot_game_dashboard(report)
plt.show()

## Chart 2: Rolling horizontal + vertical spread

In [ ]:
prepped = prep_rolling_spreads(report.rolling_spreads)
fig = plot_rolling_spreads(
    prepped,
    title=f"{PITCHER} | {PITCH_TYPE} | Game {GAME_PK} — Rolling Spread",
    window=WINDOW,
)
plt.show()

## Chart 3: Rolling miss distance

In [ ]:
prepped = prep_rolling_miss(report.rolling_miss)
fig = plot_rolling_miss(
    prepped,
    title=f"{PITCHER} | {PITCH_TYPE} | Game {GAME_PK} — Rolling Miss Distance",
    window=WINDOW,
)
plt.show()

## Chart 4: Location hub cluster map

In [ ]:
prepped = prep_cluster_map(
    report.game_with_clusters, report.cluster_model.centers
)
fig = plot_cluster_map(
    prepped,
    title=f"{PITCHER} | {PITCH_TYPE} | Location Hubs (k={report.cluster_model.k})"
)
plt.show()

## Chart 5: Miss vectors (directional arrows)

In [ ]:
prepped = prep_miss_vectors(
    report.game_with_clusters, report.cluster_model.centers, max_pitches=80
)
fig = plot_miss_vectors(
    prepped,
    title=f"{PITCHER} | {PITCH_TYPE} | Miss Vectors from Location Hubs"
)
plt.show()

## Chart 6: Cluster drift (early → mid → late)

In [ ]:
prepped = prep_cluster_drift(report.drift_report)
fig = plot_cluster_drift(
    prepped,
    title=f"{PITCHER} | {PITCH_TYPE} | Hub Drift Over Game"
)
plt.show()

## Chart 7: Hub assignment by pitch number

In [ ]:
prepped = prep_cluster_usage(report.game_with_clusters)
fig = plot_cluster_usage(
    prepped,
    title=f"{PITCHER} | {PITCH_TYPE} | Hub Assignment by Pitch"
)
plt.show()

## Interpretation note

The location hubs shown in these charts are **empirical cluster centers** — they represent where this pitcher *typically* throws this pitch type across the selected date range.

They are **not** direct observations of catcher targets or pitcher intent. Terminology used: *location hub*, *empirical target region*, *pitch-type location cluster*.

Increasing miss distance or visible hub drift over the course of a game may suggest command deterioration, but should be interpreted alongside context: opponent, pitch mix changes, and count situations.

---

## Part 2: Arsenal Analysis (Pitch-Type-Centric)

The charts above are organized by **cluster label** (location hubs).
The following section reorganizes everything by **pitch type** — FF, SL, CH — and adds new performance metrics:

- Rolling velocity trend (with ±1 std band and velocity drop annotation)
- Rolling strike%, zone%, and whiff% over pitch sequence
- Rolling pitch mix (usage % of each type over game)
- Pitch-type scatter in the strike zone (cluster hubs as secondary overlays)
- Full arsenal dashboard (one grid covering every pitch type)

Use `build_arsenal_report()` as the entry point — it covers all pitch types in one call.

In [ ]:
from src.services.pitcher_service import build_arsenal_report, ArsenalReport
from src.visualization.dashboards import plot_arsenal_dashboard

# Build the full arsenal report for the same game
arsenal = build_arsenal_report(
    pitcher_id=PITCHER,
    game_pk=GAME_PK,
    start_date=START_DATE,
    end_date=END_DATE,
    window=WINDOW,
    k=K,
)

if arsenal.warnings:
    for w in arsenal.warnings:
        print(f"[WARNING] {w}")

print(f"Arsenal ready: pitcher={arsenal.pitcher_id}, game={arsenal.game_pk}, date={arsenal.game_date}")
print(f"Analyzed pitch types: {arsenal.analyzed_pitch_types}")


## Chart A: Pitch type zone map

Single strike-zone scatter with **pitch type as the primary color axis**.
Cluster hub centers appear as small diamond markers — secondary reference, not the legend subject.

In [ ]:
from src.visualization.zone_plots import prep_pitch_type_zone_map, plot_pitch_type_zone_map

prepped_zone = prep_pitch_type_zone_map(
    arsenal.game_df,
    game_pk=GAME_PK,
    cluster_models=arsenal.cluster_models,
)
fig = plot_pitch_type_zone_map(
    prepped_zone,
    title=f"{PITCHER} | Game {GAME_PK} — All Pitch Types",
)
plt.show()

## Chart B: Arsenal zone overview (one subplot per pitch type)

In [ ]:
from src.visualization.zone_plots import plot_arsenal_zone_overview

fig = plot_arsenal_zone_overview(
    arsenal.game_df,
    game_pk=GAME_PK,
    cluster_models=arsenal.cluster_models,
    max_pitch_types=4,
    title=f"{PITCHER} | Game {GAME_PK} — Zone Overview by Pitch Type",
)
plt.show()

## Chart C: Rolling velocity trend

Rolling mean ± 1 std shading. Annotates velocity drop from the first 10 pitches to the last 10 — a quick flag for arm fatigue.

In [ ]:
from src.visualization.rolling_plots import prep_velocity_trend, plot_velocity_trend

# Show for the primary pitch type (first in arsenal)
primary_pt = arsenal.analyzed_pitch_types[0]
velo_df = arsenal.performance_metrics[primary_pt]["velocity"]

prepped_velo = prep_velocity_trend(velo_df)
fig = plot_velocity_trend(
    prepped_velo,
    title=f"{PITCHER} | {primary_pt} | Game {GAME_PK} — Velocity Trend",
    window=WINDOW,
)
plt.show()

## Chart D: Rolling performance panel (strike%, zone%, whiff%)

Three rates on one chart. Strike% in solid blue, zone% dashed green, whiff% dotted orange.
Each is a rolling mean computed over `WINDOW` pitches.

In [ ]:
from src.visualization.rolling_plots import prep_performance_panel, plot_performance_panel

perf_prepped = prep_performance_panel(arsenal.performance_metrics[primary_pt])
fig = plot_performance_panel(
    perf_prepped,
    title=f"{PITCHER} | {primary_pt} | Game {GAME_PK} — Performance Rates",
    window=WINDOW,
)
plt.show()

## Chart E: Pitch mix trend

Rolling usage % of each pitch type across the game sequence.
Uses a wider `PITCH_MIX_WINDOW_DEFAULT` (15 pitches) because individual type counts are sparse in a narrow window.

In [ ]:
from src.visualization.rolling_plots import prep_pitch_mix_trend, plot_pitch_mix_trend

mix_prepped = prep_pitch_mix_trend(arsenal.pitch_mix_df)
fig = plot_pitch_mix_trend(
    mix_prepped,
    title=f"{PITCHER} | Game {GAME_PK} — Rolling Pitch Mix",
)
plt.show()

## Chart F: Full arsenal dashboard

All-in-one grid: pitch mix + overall spread in the header row, then one row per pitch type showing zone scatter, velocity, performance rates, and miss distance.

This is the primary summary view for `build_arsenal_report()` output.

In [ ]:
fig = plot_arsenal_dashboard(
    arsenal.game_df,
    game_pk=GAME_PK,
    cluster_models=arsenal.cluster_models,
    window=WINDOW,
    max_pitch_types=4,
)
plt.show()